# A3: Centralized DP-Compatible Logistic-Regression Baseline

## 1. Experimental Setup

This experiment uses the reduced A2 representation as the starting point for a centralized gradient-based baseline that can later be reproduced under federated and differentially private training.

The A2 cohort, LOS definition, and two-stage patient-level partitioning are retained unchanged. Before model training, a feature-hygiene audit is applied only to the inner training partition to identify computationally invalid or redundant predictors. No LOS labels or held-out observations are used to determine feature removal.


In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

SEED = 42
PROJECT_DIR = os.environ.get("MIMIC_FAIRNESS_WORKDIR")

if not PROJECT_DIR:
    raise RuntimeError(
        "Set MIMIC_FAIRNESS_WORKDIR to your private research workspace."
    )

A2_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "laboratory_feature_reduction",
)

A2_CSV = os.path.join(
    A2_DIR,
    "icu_data_no_imputation_atc3_lab5pct.csv",
)

HYGIENE_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "feature_hygiene",
)

LR_RESULTS_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "centralized_logistic_regression",
)

os.makedirs(HYGIENE_DIR, exist_ok=True)
os.makedirs(LR_RESULTS_DIR, exist_ok=True)

model_data = pd.read_csv(A2_CSV, low_memory=False)

# Preserve the LOS definition used in A0--A2.
model_data["los_type"] = np.where(
    model_data["los"] > 4,
    "Long",
    "Short",
)

print("A2 dataset shape:", model_data.shape)
print("ICU stays:", model_data["stay_id"].nunique())
print("Patients:", model_data["subject_id"].nunique())

# Reproduce the established patient-level splits.
patient_ids = model_data["subject_id"].unique()

train_patient_ids, validation_patient_ids = train_test_split(
    patient_ids,
    test_size=0.2,
    random_state=SEED,
)

train_data = model_data[
    model_data["subject_id"].isin(train_patient_ids)
].copy()

validation_data = model_data[
    model_data["subject_id"].isin(validation_patient_ids)
].copy()

inner_patient_ids = train_data["subject_id"].unique()

train_train_ids, test_train_ids = train_test_split(
    inner_patient_ids,
    test_size=0.2,
    random_state=SEED,
)

train_train_data = train_data[
    train_data["subject_id"].isin(train_train_ids)
].copy()

test_train_data = train_data[
    train_data["subject_id"].isin(test_train_ids)
].copy()

assert len(train_train_data) == 47543
assert len(test_train_data) == 11972
assert len(validation_data) == 14986

print()
print("Inner training rows:", len(train_train_data))
print("Inner testing rows:", len(test_train_data))
print("Held-out rows:", len(validation_data))

exclude_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "first_careunit",
    "last_careunit",
    "intime",
    "outtime",
    "los",
    "los_type",
    "age",
    "gender",
    "insurance",
    "race",
    "PHYSICIAN REFERRAL",
]

feature_columns = [
    column
    for column in train_data.columns
    if column not in exclude_columns
]

numeric_columns = (
    train_train_data[feature_columns]
    .select_dtypes(include=["float64", "int64"])
    .columns
    .tolist()
)

assert len(numeric_columns) == 353

X_hygiene = train_train_data[numeric_columns]

print()
print("Candidate features:", len(feature_columns))
print("Numeric model features:", len(numeric_columns))
print("Audit rows:", len(X_hygiene))


## 2. Feature Hygiene Audit

The 353 numeric A2 predictors are audited on the inner training partition for missing or infinite values, constant and highly concentrated features, and exact duplicate columns.

The near-constant flag is used only for inspection. Nonconstant sparse features are not removed on the basis of this diagnostic because doing so would introduce an additional feature-selection rule beyond the medication and laboratory reduction procedures.


In [ ]:
import hashlib
from collections import defaultdict

audit_rows = []

for column in numeric_columns:
    values = X_hygiene[column].to_numpy(dtype=np.float64)

    missing_count = int(np.isnan(values).sum())
    infinite_count = int(np.isinf(values).sum())
    finite_values = values[np.isfinite(values)]

    if len(finite_values) > 0:
        finite_series = pd.Series(finite_values)
        unique_count = int(finite_series.nunique())
        dominant_fraction = float(
            finite_series.value_counts(normalize=True).iloc[0]
        )
        zero_fraction = float(np.mean(finite_values == 0))
        minimum = float(np.min(finite_values))
        maximum = float(np.max(finite_values))
        mean = float(np.mean(finite_values))
        std = float(np.std(finite_values))
    else:
        unique_count = 0
        dominant_fraction = np.nan
        zero_fraction = np.nan
        minimum = np.nan
        maximum = np.nan
        mean = np.nan
        std = np.nan

    audit_rows.append({
        "feature": column,
        "missing_count": missing_count,
        "infinite_count": infinite_count,
        "unique_values": unique_count,
        "dominant_fraction": dominant_fraction,
        "zero_fraction": zero_fraction,
        "minimum": minimum,
        "maximum": maximum,
        "mean": mean,
        "std": std,
        "constant": unique_count <= 1,
        "near_constant_99_5pct": (
            dominant_fraction >= 0.995
            if not np.isnan(dominant_fraction)
            else False
        ),
    })

feature_hygiene = pd.DataFrame(audit_rows)

print("Feature hygiene audit")
print("---------------------")
print("Features audited:", len(feature_hygiene))
print(
    "Features containing missing values:",
    int((feature_hygiene["missing_count"] > 0).sum()),
)
print(
    "Features containing infinite values:",
    int((feature_hygiene["infinite_count"] > 0).sum()),
)
print(
    "Constant features:",
    int(feature_hygiene["constant"].sum()),
)
print(
    "Near-constant features (>=99.5% one value):",
    int(feature_hygiene["near_constant_99_5pct"].sum()),
)

# Identify exact duplicate columns.
hash_groups = defaultdict(list)

for column in numeric_columns:
    values = X_hygiene[column].to_numpy(
        dtype=np.float64,
        copy=True,
    )
    digest = hashlib.sha256(values.tobytes()).hexdigest()
    hash_groups[digest].append(column)

duplicate_pairs = []

for columns in hash_groups.values():
    if len(columns) > 1:
        for i in range(len(columns)):
            for j in range(i + 1, len(columns)):
                c1, c2 = columns[i], columns[j]

                if np.array_equal(
                    X_hygiene[c1].to_numpy(dtype=np.float64),
                    X_hygiene[c2].to_numpy(dtype=np.float64),
                    equal_nan=True,
                ):
                    duplicate_pairs.append({
                        "feature_1": c1,
                        "feature_2": c2,
                    })

duplicate_features = pd.DataFrame(
    duplicate_pairs,
    columns=["feature_1", "feature_2"],
)

print()
print(
    "Exact duplicate feature pairs:",
    len(duplicate_features),
)

if len(duplicate_features):
    display(duplicate_features)

HYGIENE_CSV = os.path.join(
    HYGIENE_DIR,
    "a2_feature_hygiene_audit.csv",
)

DUPLICATES_CSV = os.path.join(
    HYGIENE_DIR,
    "a2_exact_duplicate_features.csv",
)

feature_hygiene.to_csv(HYGIENE_CSV, index=False)
duplicate_features.to_csv(DUPLICATES_CSV, index=False)


## 3. Hygiene Audit Outcome

The audit identified no missing or infinite values. Two ATC3 medication variables were constant and equal to zero throughout the inner training partition. They also formed the only exact duplicate pair and were removed because they contain no information for model estimation.

The remaining highly sparse but nonconstant predictors were retained. Accordingly, the hygiene step reduces the fixed representation from 353 to 351 predictors without introducing an additional prevalence-based feature-selection criterion.


In [ ]:
constant_features = (
    feature_hygiene.loc[
        feature_hygiene["constant"],
        "feature",
    ]
    .tolist()
)

print("Constant features:", constant_features)

assert set(constant_features) == {
    "med_atc3_V08A",
    "med_atc3_V08C",
}

final_feature_columns = [
    column
    for column in numeric_columns
    if column not in constant_features
]

assert len(final_feature_columns) == 351

X_final = train_train_data[final_feature_columns]
duplicate_columns_after_hygiene = X_final.T.duplicated()

assert int(duplicate_columns_after_hygiene.sum()) == 0

print()
print("Final feature-space audit")
print("-------------------------")
print("A2 numeric predictors:", len(numeric_columns))
print("Constant features removed:", len(constant_features))
print("Final predictors:", len(final_feature_columns))
print(
    "Exact duplicate columns after hygiene:",
    int(duplicate_columns_after_hygiene.sum()),
)

FINAL_FEATURES_CSV = os.path.join(
    HYGIENE_DIR,
    "final_351_feature_list.csv",
)

pd.DataFrame({
    "feature": final_feature_columns
}).to_csv(
    FINAL_FEATURES_CSV,
    index=False,
)

print("Saved:", FINAL_FEATURES_CSV)


## 4. Fixed Preprocessing for Gradient-Based and DP Experiments

An initial preprocessing diagnostic showed that indiscriminate standardization was unsuitable for this representation: very sparse binary indicators were inflated substantially, while several continuous clinical variables contained extreme numerical values.

The final preprocessing therefore treats binary and continuous predictors separately. Binary predictors remain in their original \(0/1\) representation. Continuous predictors use the bounded signed-log transformation

\[
u = \operatorname{sign}(x)\log(1+|x|),
\qquad
x' = \frac{u}{1+|u|}.
\]

The transformation preserves zero, compresses extreme values, and bounds transformed continuous inputs to \((-1,1)\). Once the feature-type specification is fixed, the transformation itself contains no fitted parameters. The resulting preprocessing specification is frozen for the subsequent centralized, federated, and differentially private model experiments.


In [ ]:
# Separate binary and continuous predictors using the fixed inner-training representation.
binary_features = []
continuous_features = []

for feature in final_feature_columns:
    values = train_train_data[feature].dropna().unique()

    if set(values).issubset({0, 1, 0.0, 1.0}):
        binary_features.append(feature)
    else:
        continuous_features.append(feature)

assert (
    len(binary_features) + len(continuous_features)
    == len(final_feature_columns)
)

print("Final predictors:", len(final_feature_columns))
print("Binary predictors:", len(binary_features))
print("Continuous predictors:", len(continuous_features))


def bounded_signed_log_transform(
    data,
    feature_columns,
    continuous_columns,
):
    """Bound continuous predictors while leaving binary predictors unchanged."""

    X = data[feature_columns].copy().astype(np.float32)

    values = X[continuous_columns].to_numpy(
        dtype=np.float64
    )

    signed_log = (
        np.sign(values)
        * np.log1p(np.abs(values))
    )

    transformed = (
        signed_log
        / (1.0 + np.abs(signed_log))
    )

    X.loc[:, continuous_columns] = (
        transformed.astype(np.float32)
    )

    return X


X_train_final = bounded_signed_log_transform(
    train_train_data,
    final_feature_columns,
    continuous_features,
)

X_test_final = bounded_signed_log_transform(
    test_train_data,
    final_feature_columns,
    continuous_features,
)

X_validation_final = bounded_signed_log_transform(
    validation_data,
    final_feature_columns,
    continuous_features,
)

total_nonfinite = (
    (~np.isfinite(X_train_final.to_numpy())).sum()
    + (~np.isfinite(X_test_final.to_numpy())).sum()
    + (~np.isfinite(X_validation_final.to_numpy())).sum()
)

max_abs_continuous = (
    X_train_final[continuous_features]
    .abs()
    .to_numpy()
    .max()
)

assert X_train_final.shape == (47543, 351)
assert X_test_final.shape == (11972, 351)
assert X_validation_final.shape == (14986, 351)
assert max_abs_continuous < 1
assert set(
    np.unique(
        X_train_final[binary_features].to_numpy()
    )
).issubset({0.0, 1.0})
assert total_nonfinite == 0

print()
print("DP-compatible preprocessing audit")
print("---------------------------------")
print("Training shape:", X_train_final.shape)
print("Testing shape:", X_test_final.shape)
print("Held-out shape:", X_validation_final.shape)
print(
    "Maximum absolute continuous value:",
    max_abs_continuous,
)
print(
    "Maximum binary value:",
    X_train_final[binary_features]
    .to_numpy()
    .max(),
)
print(
    "Minimum binary value:",
    X_train_final[binary_features]
    .to_numpy()
    .min(),
)
print("Non-finite values:", total_nonfinite)

preprocessing_spec = pd.DataFrame({
    "feature": final_feature_columns,
    "feature_type": [
        "binary"
        if feature in binary_features
        else "continuous"
        for feature in final_feature_columns
    ],
    "transformation": [
        "identity_0_1"
        if feature in binary_features
        else "bounded_signed_log"
        for feature in final_feature_columns
    ],
})

PREPROCESSING_SPEC = os.path.join(
    HYGIENE_DIR,
    "a3_preprocessing_specification.csv",
)

preprocessing_spec.to_csv(
    PREPROCESSING_SPEC,
    index=False,
)

print()
print("Preprocessing specification")
print("---------------------------")
print("Total features:", len(preprocessing_spec))
print(
    "Binary features:",
    (preprocessing_spec["feature_type"] == "binary").sum(),
)
print(
    "Continuous features:",
    (preprocessing_spec["feature_type"] == "continuous").sum(),
)
print("Saved:", PREPROCESSING_SPEC)


## 5. Centralized Logistic-Regression Baseline

A centralized logistic-regression classifier is trained using the fixed 351-feature representation and the predefined bounded preprocessing procedure.

The model is implemented in PyTorch as a single linear layer with one output logit. No differential privacy is applied in this experiment. The objective is to establish a non-private centralized reference using the same model architecture and input representation that can subsequently be trained in federated and DP-SGD settings.


In [ ]:
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# Labels.
y_train_lr = (
    train_train_data["los_type"]
    .map({"Short": 0, "Long": 1})
    .astype(np.float32)
    .to_numpy()
)

y_test_lr = (
    test_train_data["los_type"]
    .map({"Short": 0, "Long": 1})
    .astype(np.float32)
    .to_numpy()
)

y_validation_lr = (
    validation_data["los_type"]
    .map({"Short": 0, "Long": 1})
    .astype(np.float32)
    .to_numpy()
)

# Tensors.
X_train_tensor = torch.tensor(
    X_train_final.to_numpy(),
    dtype=torch.float32,
)

X_test_tensor = torch.tensor(
    X_test_final.to_numpy(),
    dtype=torch.float32,
)

X_validation_tensor = torch.tensor(
    X_validation_final.to_numpy(),
    dtype=torch.float32,
)

y_train_tensor = torch.tensor(
    y_train_lr,
    dtype=torch.float32,
).unsqueeze(1)

y_test_tensor = torch.tensor(
    y_test_lr,
    dtype=torch.float32,
).unsqueeze(1)

y_validation_tensor = torch.tensor(
    y_validation_lr,
    dtype=torch.float32,
).unsqueeze(1)

print("Device:", device)
print(
    "Training:",
    X_train_tensor.shape,
    y_train_tensor.shape,
)
print(
    "Testing:",
    X_test_tensor.shape,
    y_test_tensor.shape,
)
print(
    "Held-out:",
    X_validation_tensor.shape,
    y_validation_tensor.shape,
)

print()
print("Training class distribution")
print("---------------------------")
print("Short:", int((y_train_lr == 0).sum()))
print("Long:", int((y_train_lr == 1).sum()))
print(
    "Long proportion:",
    f"{(y_train_lr == 1).mean():.4f}",
)


class LogisticRegressionModel(nn.Module):

    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)


model_lr = LogisticRegressionModel(
    n_features=len(final_feature_columns)
).to(device)

n_parameters = sum(
    parameter.numel()
    for parameter in model_lr.parameters()
    if parameter.requires_grad
)

assert n_parameters == 352

print()
print(model_lr)
print("Input features:", len(final_feature_columns))
print("Trainable parameters:", n_parameters)

BATCH_SIZE = 256
LEARNING_RATE = 0.05
EPOCHS = 30

generator = torch.Generator()
generator.manual_seed(SEED)

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(
    model_lr.parameters(),
    lr=LEARNING_RATE,
)

sampling_rate = BATCH_SIZE / len(train_dataset)

print()
print("Training configuration")
print("----------------------")
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", EPOCHS)
print(
    "Approximate sampling rate:",
    f"{sampling_rate:.6f}",
)


In [ ]:
training_history = []

for epoch in range(1, EPOCHS + 1):
    model_lr.train()

    total_loss = 0.0
    total_examples = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model_lr(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    mean_loss = total_loss / total_examples

    training_history.append({
        "epoch": epoch,
        "loss": mean_loss,
    })

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d}/{EPOCHS}"
            f" - loss: {mean_loss:.6f}"
        )

training_history_df = pd.DataFrame(training_history)

training_history_df.to_csv(
    os.path.join(
        LR_RESULTS_DIR,
        "logistic_regression_training_history.csv",
    ),
    index=False,
)

print("Training history saved.")


## 6. Inner-Test Evaluation

The trained logistic-regression model is first evaluated on the established inner test partition using the fixed 0.5 classification threshold. This evaluation is performed before inspecting the outer held-out cohort.


In [ ]:
model_lr.eval()

with torch.no_grad():
    logits_test = model_lr(
        X_test_tensor.to(device)
    )

    test_prob = (
        torch.sigmoid(logits_test)
        .cpu()
        .numpy()
        .ravel()
    )

test_pred = (test_prob > 0.5).astype(int)

lr_test_roc = roc_auc_score(
    y_test_lr,
    test_prob,
)

lr_test_pr = average_precision_score(
    y_test_lr,
    test_prob,
)

lr_test_accuracy = accuracy_score(
    y_test_lr,
    test_pred,
)

lr_test_cm = confusion_matrix(
    y_test_lr,
    test_pred,
)

print("Centralized logistic regression: inner test")
print("-------------------------------------------")
print(f"AUC-ROC: {lr_test_roc:.4f}")
print(f"AUC-PR:  {lr_test_pr:.4f}")
print(f"Accuracy: {lr_test_accuracy:.4f}")
print("Confusion Matrix:")
print(lr_test_cm)

print()
print("Classification Report:")
print(
    classification_report(
        y_test_lr,
        test_pred,
        target_names=["Short", "Long"],
        digits=4,
    )
)


## 7. Held-Out Evaluation

The finalized non-private logistic-regression model is evaluated on the outer held-out patient partition without retraining, preprocessing changes, class reweighting, or decision-threshold adjustment. The fixed 0.5 threshold is retained.


In [ ]:
model_lr.eval()

with torch.no_grad():
    validation_logits_lr = model_lr(
        X_validation_tensor.to(device)
    )

    validation_prob_lr = (
        torch.sigmoid(validation_logits_lr)
        .cpu()
        .numpy()
        .ravel()
    )

validation_pred_lr = (
    validation_prob_lr > 0.5
).astype(int)

lr_validation_roc = roc_auc_score(
    y_validation_lr,
    validation_prob_lr,
)

lr_validation_pr = average_precision_score(
    y_validation_lr,
    validation_prob_lr,
)

lr_validation_accuracy = accuracy_score(
    y_validation_lr,
    validation_pred_lr,
)

lr_validation_cm = confusion_matrix(
    y_validation_lr,
    validation_pred_lr,
)

print("Centralized logistic regression: held-out evaluation")
print("----------------------------------------------------")
print(f"AUC-ROC: {lr_validation_roc:.4f}")
print(f"AUC-PR:  {lr_validation_pr:.4f}")
print(f"Accuracy: {lr_validation_accuracy:.4f}")
print("Confusion Matrix:")
print(lr_validation_cm)

print()
print("Classification Report:")
print(
    classification_report(
        y_validation_lr,
        validation_pred_lr,
        target_names=["Short", "Long"],
        digits=4,
    )
)

lr_heldout_summary = pd.DataFrame([{
    "model": "Centralized Logistic Regression",
    "features": len(final_feature_columns),
    "trainable_parameters": n_parameters,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "roc_auc": lr_validation_roc,
    "pr_auc": lr_validation_pr,
    "accuracy": lr_validation_accuracy,
    "TN": lr_validation_cm[0, 0],
    "FP": lr_validation_cm[0, 1],
    "FN": lr_validation_cm[1, 0],
    "TP": lr_validation_cm[1, 1],
}])

lr_heldout_summary.to_csv(
    os.path.join(
        LR_RESULTS_DIR,
        "logistic_regression_heldout_summary.csv",
    ),
    index=False,
)

display(lr_heldout_summary)


## 8. Subgroup Performance Evaluation

The same held-out predictions are evaluated separately across insurance and race groups. No subgroup-specific models are trained. Long-stay precision, recall, F1-score, false-positive rate, and accuracy are reported descriptively.


In [ ]:
lr_fairness_data = pd.DataFrame({
    "Actual": y_validation_lr.astype(int),
    "Predicted": validation_pred_lr.astype(int),
    "Insurance": validation_data["insurance"].to_numpy(),
    "Race": validation_data["race"].to_numpy(),
})

assert len(lr_fairness_data) == 14986


def subgroup_summary(data, group_column):
    rows = []

    for group, subset in data.groupby(
        group_column,
        dropna=False,
    ):
        y_true = subset["Actual"].astype(int)
        y_pred = subset["Predicted"].astype(int)

        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        )

        tn, fp, fn, tp = cm.ravel()

        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else np.nan
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else np.nan
        )

        f1 = (
            2 * precision * recall
            / (precision + recall)
            if (precision + recall) > 0
            else np.nan
        )

        fpr = (
            fp / (fp + tn)
            if (fp + tn) > 0
            else np.nan
        )

        rows.append({
            group_column: group,
            "n": len(subset),
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "accuracy": (tp + tn) / len(subset),
            "long_precision": precision,
            "long_recall": recall,
            "long_f1": f1,
            "TPR": recall,
            "FPR": fpr,
            "positive_prediction_rate":
                (tp + fp) / len(subset),
        })

    return (
        pd.DataFrame(rows)
        .sort_values("n", ascending=False)
        .reset_index(drop=True)
    )


lr_insurance_fairness = subgroup_summary(
    lr_fairness_data.dropna(
        subset=["Insurance"]
    ),
    "Insurance",
)

lr_race_fairness = subgroup_summary(
    lr_fairness_data.dropna(
        subset=["Race"]
    ),
    "Race",
)

print(
    "Centralized logistic regression: "
    "Insurance subgroup performance"
)

display(
    lr_insurance_fairness[
        [
            "Insurance",
            "n",
            "long_precision",
            "long_recall",
            "long_f1",
            "FPR",
            "accuracy",
        ]
    ]
)

print(
    "Centralized logistic regression: "
    "Race subgroup performance"
)

display(
    lr_race_fairness[
        [
            "Race",
            "n",
            "long_precision",
            "long_recall",
            "long_f1",
            "FPR",
            "accuracy",
        ]
    ]
)

lr_insurance_fairness.to_csv(
    os.path.join(
        LR_RESULTS_DIR,
        "logistic_regression_insurance_subgroups.csv",
    ),
    index=False,
)

lr_race_fairness.to_csv(
    os.path.join(
        LR_RESULTS_DIR,
        "logistic_regression_race_subgroups.csv",
    ),
    index=False,
)


In [ ]:
def disparity_spread(
    dataframe,
    minimum_n=100,
):
    eligible = dataframe[
        dataframe["n"] >= minimum_n
    ].copy()

    return {
        "groups_included": len(eligible),
        "long_recall_spread":
            eligible["long_recall"].max()
            - eligible["long_recall"].min(),
        "FPR_spread":
            eligible["FPR"].max()
            - eligible["FPR"].min(),
    }


lr_insurance_spread = disparity_spread(
    lr_insurance_fairness
)

lr_race_spread = disparity_spread(
    lr_race_fairness
)

lr_disparity_summary = pd.DataFrame([
    {
        "attribute": "Insurance",
        **lr_insurance_spread,
    },
    {
        "attribute": "Race",
        **lr_race_spread,
    },
])

print(
    "Subgroup disparity summary "
    "(groups with n >= 100)"
)

display(
    lr_disparity_summary.round(4)
)

lr_disparity_summary.to_csv(
    os.path.join(
        LR_RESULTS_DIR,
        "logistic_regression_subgroup_disparity_summary.csv",
    ),
    index=False,
)


## 9. Comparison with the A2 XGBoost Reference

The centralized logistic-regression model is compared with the A2 XGBoost reference on the same held-out patient partition. The comparison is a reference comparison rather than a pure architecture-only experiment because A3 introduces the fixed bounded preprocessing used for subsequent gradient-based models.

| Metric | A2 XGBoost | Logistic Regression | Difference |
|---|---:|---:|---:|
| ROC-AUC | 0.8691 | 0.8473 | -0.0218 |
| PR-AUC | 0.7728 | 0.7371 | -0.0357 |
| Accuracy | 0.8222 | 0.8116 | -0.0106 |
| Long precision | 0.7477 | 0.7261 | -0.0216 |
| Long recall | 0.6007 | 0.5810 | -0.0197 |
| Long F1 | 0.6662 | 0.6455 | -0.0207 |

Logistic regression produces moderately lower predictive performance than XGBoost, consistent with its simpler linear decision function.

Considering only groups with at least 100 held-out observations, the subgroup-performance spreads are:

| Attribute | Model | Long-recall spread | FPR spread |
|---|---|---:|---:|
| Insurance | A2 XGBoost | 0.0349 | 0.0218 |
| Insurance | Logistic Regression | 0.0692 | 0.0363 |
| Race | A2 XGBoost | 0.1133 | 0.0258 |
| Race | Logistic Regression | 0.1264 | 0.0388 |

The logistic-regression model therefore shows somewhat wider subgroup variation in Long-stay recall and false-positive rates. These differences are interpreted descriptively rather than as evidence that either model is categorically fair or unfair.


## 10. Summary

The finalized centralized logistic-regression baseline uses 351 predictors and contains 352 trainable parameters. On the held-out cohort it achieved a ROC-AUC of 0.8473, PR-AUC of 0.7371, accuracy of 0.8116, Long-stay recall of 0.5810, and Long-stay F1-score of 0.6455.

Although its predictive performance is lower than the A2 XGBoost reference, the model is deliberately compact and uses the same fixed representation and preprocessing that can be carried forward into federated and DP-SGD experiments. It is therefore retained as the primary linear non-private baseline for subsequent privacy experiments.
